# <font color='lightgreen'>03 Build Star Schema</font> ⭐

	Construimos el esquema estrella (dimensiones y tabla de hechos) a partir de los datos limpios.
	Este notebook es **idempotente** y produce un modelo listo para cargar en DuckDB y consumir desde el dashboard.
---

### <font color='lightgreen'>Librerías</font>

In [110]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import logging
import yaml
from datetime import datetime

print("✅ Librerías cargadas correctamente")
print('Versión de pandas:', pd.__version__)
print('Versión de numpy:', np.__version__)

✅ Librerías cargadas correctamente
Versión de pandas: 3.0.2
Versión de numpy: 2.4.4


### <font color='lightgreen'>Configuración</font>

#### <font color='lightgreen'>Configuración logging</font>

In [111]:
# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

#### <font color='lightgreen'>Configuración de ruta y carga</font>

In [112]:
# Raíz del proyecto
PROJECT_ROOT = Path.cwd().parent
DATA_CLEAN = PROJECT_ROOT / "data" / "clean"
DATA_CURATED = PROJECT_ROOT / "data" / "curated"

# Crear carpeta de salida
DATA_CURATED.mkdir(parents=True, exist_ok=True)

logger.info(f"📁 Datos limpios: {DATA_CLEAN}")
logger.info(f"📁 Modelo estrella: {DATA_CURATED}")

# Cargar configuración (factores de emisión, umbrales, etc.)
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r") as f:
        config = yaml.safe_load(f)
    logger.info(f"✅ Configuración cargada desde {CONFIG_PATH}")
else:
    # Configuración por defecto
    config = {
        "factors": {"co2_per_kwh": 0.0004},
        "targets": {"reciclaje": 60, "margen_neto": 12}
    }
    logger.warning(f"⚠️ No se encontró config.yaml, usando valores por defecto")

FACTOR_EMISION = config["factors"]["co2_per_kwh"]
logger.info(f"🌍 Factor de emisión: {FACTOR_EMISION} tCO2e/kWh")

2026-04-15 19:35:06,139 - INFO - 📁 Datos limpios: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\clean
2026-04-15 19:35:06,143 - INFO - 📁 Modelo estrella: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\curated
2026-04-15 19:35:06,155 - INFO - ✅ Configuración cargada desde c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\config.yaml
2026-04-15 19:35:06,156 - INFO - 🌍 Factor de emisión: 0.0004 tCO2e/kWh


### <font color='lightgreen'>3. Funciones auxiliares y modulares</font>

In [113]:
def cargar_datos_limpios(nombre_archivo):
    """Carga un CSV limpio y convierte automáticamente las columnas de fecha."""
    file_path = DATA_CLEAN / f"{nombre_archivo}_clean.csv"
    if not file_path.exists():
        logger.error(f"❌ No se encuentra {file_path}")
        return None
    df = pd.read_csv(file_path, encoding='utf-8')
    # Intentar convertir columnas que parecen fechas (por nombre o por contenido)
    for col in df.columns:
        if 'fecha' in col.lower() or 'mes' in col.lower():
            df[col] = pd.to_datetime(df[col], errors='coerce')
    logger.info(f"   Cargado {nombre_archivo}: {len(df):,} filas")
    return df

def crear_dimension_unica(df, columna, nombre_dim, id_name='id'):
    """Crea una dimensión a partir de valores únicos de una columna."""
    valores = df[columna].dropna().unique()
    dim = pd.DataFrame({columna: sorted(valores)})
    dim.insert(0, id_name, range(1, len(dim) + 1))
    logger.info(f"   ✅ {nombre_dim}: {len(dim)} registros únicos")
    return dim

def validar_fechas(df, columna):
    """Convierte a datetime y registra cuántas quedaron nulas."""
    antes = df[columna].isna().sum()
    df[columna] = pd.to_datetime(df[columna], errors='coerce')
    despues = df[columna].isna().sum()
    if despues > antes:
        logger.warning(f"   ⚠️ {columna}: {despues - antes} fechas inválidas convertidas a NaT")
    return df

### <font color='lightgreen'>4. Carga todos los datsasets limpios</font>

In [114]:
logger.info("="*50)
logger.info("CARGANDO DATOS LIMPIOS")
logger.info("="*50)

datasets = [
    "compras", "consumo_energetico", "encuestas",
    "eventos_rrhh", "personal_nomina", "residuos", "ventas"
]

datos = {}
for ds in datasets:
    df = cargar_datos_limpios(ds)
    if df is not None:
        datos[ds] = df

2026-04-15 19:35:06,270 - INFO - ==================================================
2026-04-15 19:35:06,271 - INFO - CARGANDO DATOS LIMPIOS
2026-04-15 19:35:06,273 - INFO - ==================================================
2026-04-15 19:35:06,371 - INFO -    Cargado compras: 864 filas
2026-04-15 19:35:06,378 - INFO -    Cargado consumo_energetico: 178 filas
2026-04-15 19:35:06,387 - INFO -    Cargado encuestas: 189 filas
2026-04-15 19:35:06,397 - INFO -    Cargado eventos_rrhh: 537 filas
2026-04-15 19:35:06,410 - INFO -    Cargado personal_nomina: 1,378 filas
2026-04-15 19:35:06,426 - INFO -    Cargado residuos: 2,508 filas
2026-04-15 19:35:06,554 - INFO -    Cargado ventas: 43,893 filas


### <font color='lightgreen'>5. Crear dimensiones</font>

#### <font color='lightgreen'>5.1 Dimensión tiempo</font>

In [115]:
logger.info("\n" + "="*50)
logger.info("CREANDO DIMENSIÓN TIEMPO")
logger.info("="*50)

# Recolectar todas las fechas de todas las columnas datetime
todas_fechas = []
fechas_descartadas = 0

for df in datos.values():
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            # Extraer fechas no nulas
            fechas = df[col].dropna()
            # Filtrar fechas válidas (a partir del año 2000)
            fechas_validas = fechas[fechas >= pd.Timestamp('2000-01-01')]
            fechas_descartadas += len(fechas) - len(fechas_validas)
            todas_fechas.extend(fechas_validas.tolist())

logger.info(f"   Fechas totales recolectadas (válidas): {len(todas_fechas)}")
logger.info(f"   Fechas descartadas por ser anteriores a 2000: {fechas_descartadas}")

# Eliminar duplicados y ordenar
fechas_unicas = sorted(set(todas_fechas))
logger.info(f"   Fechas únicas después de deduplicación: {len(fechas_unicas)}")

# Crear DataFrame de dimensión
dim_tiempo = pd.DataFrame({'fecha': fechas_unicas})
dim_tiempo['id_tiempo'] = dim_tiempo['fecha'].dt.strftime('%Y%m%d').astype(int)
dim_tiempo['anio'] = dim_tiempo['fecha'].dt.year
dim_tiempo['mes'] = dim_tiempo['fecha'].dt.month
dim_tiempo['dia'] = dim_tiempo['fecha'].dt.day
dim_tiempo['trimestre'] = dim_tiempo['fecha'].dt.quarter
dim_tiempo['semana'] = dim_tiempo['fecha'].dt.isocalendar().week
dim_tiempo['nombre_mes'] = dim_tiempo['fecha'].dt.strftime('%B')
dim_tiempo['dia_semana'] = dim_tiempo['fecha'].dt.day_name()
dim_tiempo['es_finde'] = (dim_tiempo['fecha'].dt.dayofweek >= 5).astype(int)

# Reordenar columnas
dim_tiempo = dim_tiempo[['id_tiempo', 'fecha', 'anio', 'mes', 'dia', 'trimestre',
                        'semana', 'nombre_mes', 'dia_semana', 'es_finde']]

# Eliminar cualquier duplicado residual (por si acaso)
dim_tiempo = dim_tiempo.drop_duplicates(subset=['id_tiempo'])

# Verificar que no quede ninguna fecha antigua
dim_tiempo = dim_tiempo[dim_tiempo['fecha'] >= '2000-01-01']

logger.info(f"✅ dim_tiempo: {len(dim_tiempo)} fechas únicas")
if len(dim_tiempo) > 0:
    logger.info(f"   Rango: {dim_tiempo['fecha'].min().date()} → {dim_tiempo['fecha'].max().date()}")
else:
    logger.error("❌ No se generó ninguna fecha válida. Revisar datos de origen.")

2026-04-15 19:35:06,600 - INFO - 
2026-04-15 19:35:06,602 - INFO - CREANDO DIMENSIÓN TIEMPO
2026-04-15 19:35:06,602 - INFO - ==================================================
2026-04-15 19:35:06,669 - INFO -    Fechas totales recolectadas (válidas): 51789
2026-04-15 19:35:06,670 - INFO -    Fechas descartadas por ser anteriores a 2000: 48683
2026-04-15 19:35:06,702 - INFO -    Fechas únicas después de deduplicación: 2641
2026-04-15 19:35:06,766 - INFO - ✅ dim_tiempo: 2641 fechas únicas
2026-04-15 19:35:06,768 - INFO -    Rango: 2018-01-01 → 2025-12-31


#### <font color='lightgreen'>5.2 Dimensión área</font>

In [116]:
logger.info("\n" + "="*50)
logger.info("CREANDO DIMENSIÓN ÁREA")
logger.info("="*50)

areas = pd.concat([
    datos['personal_nomina'][['area']],
    datos['encuestas'][['area']],
    datos['eventos_rrhh'][['area']]
]).dropna()
dim_area = crear_dimension_unica(areas, 'area', 'dim_area', id_name='id_area')
# Luego renombrar la columna 'area' a 'nombre_area'
dim_area = dim_area.rename(columns={'area': 'nombre_area'})


2026-04-15 19:35:06,787 - INFO - 
2026-04-15 19:35:06,789 - INFO - CREANDO DIMENSIÓN ÁREA
2026-04-15 19:35:06,790 - INFO - ==================================================
2026-04-15 19:35:06,800 - INFO -    ✅ dim_area: 5 registros únicos


#### <font color='lightgreen'>5.3 Dimensión empleado</font>

In [117]:
logger.info("\n" + "="*50)
logger.info("CREANDO DIMENSIÓN EMPLEADO")
logger.info("="*50)

df_nomina = datos['personal_nomina']
cols_empleado = ['id_empleado', 'nombre', 'area', 'genero', 'fecha_ingreso']
cols_existentes = [c for c in cols_empleado if c in df_nomina.columns]
dim_empleado = df_nomina[cols_existentes].drop_duplicates(subset=['id_empleado']).copy()
dim_empleado = dim_empleado.sort_values('id_empleado').reset_index(drop=True)
logger.info(f"✅ dim_empleado: {len(dim_empleado)} empleados únicos")

2026-04-15 19:35:06,826 - INFO - 
2026-04-15 19:35:06,827 - INFO - CREANDO DIMENSIÓN EMPLEADO
2026-04-15 19:35:06,829 - INFO - ==================================================
2026-04-15 19:35:06,836 - INFO - ✅ dim_empleado: 25 empleados únicos


#### <font color='lightgreen'>5.4 Dimensión proveedor</font>

In [118]:
logger.info("\n" + "="*50)
logger.info("CREANDO DIMENSIÓN PROVEEDOR")
logger.info("="*50)
dim_proveedor = crear_dimension_unica(datos['compras'], 'proveedor', 'dim_proveedor', id_name='id_proveedor')
dim_proveedor = dim_proveedor.rename(columns={'proveedor': 'nombre_proveedor'})

2026-04-15 19:35:06,889 - INFO - 
2026-04-15 19:35:06,891 - INFO - CREANDO DIMENSIÓN PROVEEDOR
2026-04-15 19:35:06,894 - INFO - ==================================================
2026-04-15 19:35:06,897 - INFO -    ✅ dim_proveedor: 5 registros únicos


#### <font color='lightgreen'>5.5 Dimensión categoría (producto)</font>

In [119]:
logger.info("\n" + "="*50)
logger.info("CREANDO DIMENSIÓN CATEGORÍA")
logger.info("="*50)
dim_categoria = crear_dimension_unica(datos['ventas'], 'categoria', 'dim_categoria', id_name='id_categoria')
dim_categoria = dim_categoria.rename(columns={'categoria': 'nombre_categoria'})

2026-04-15 19:35:06,993 - INFO - 
2026-04-15 19:35:06,994 - INFO - CREANDO DIMENSIÓN CATEGORÍA
2026-04-15 19:35:06,994 - INFO - ==================================================
2026-04-15 19:35:07,001 - INFO -    ✅ dim_categoria: 10 registros únicos


#### <font color='lightgreen'>5.6 Dimensión Métrica (catálogo de KPIs)</font>

In [120]:
logger.info("\n" + "="*50)
logger.info("CREANDO DIMENSIÓN MÉTRICA (ampliada)")
logger.info("="*50)

metricas = [
    # id, nombre, categoria, subcategoria, unidad, formula, prioridad, frecuencia_recomendada, tipo_agregacion, tendencia_deseada, descripcion
    (1, 'Ingresos', 'Financiera', 'Ventas', 'ARS', 'Suma de ventas', 'Crítica', 'Mensual', 'SUM', 'UP', 'Ingresos totales por ventas'),
    (2, 'Costo Compras', 'Financiera', 'Insumos', 'ARS', 'Suma de compras', 'Importante', 'Mensual', 'SUM', 'DOWN', 'Costos de adquisición'),
    (3, 'Gasto Personal', 'Financiera', 'RRHH', 'ARS', 'Suma de nóminas', 'Importante', 'Mensual', 'SUM', 'DOWN', 'Salarios y bonificaciones'),
    (4, 'Consumo Energético', 'E', 'Energía', 'kWh', 'Suma de kWh consumidos', 'Crítica', 'Mensual', 'SUM', 'DOWN', 'Electricidad + gas (convertido)'),
    (5, 'Tasa Reciclaje', 'E', 'Residuos', '%', '(kg_reciclados/kg_generados)*100', 'Importante', 'Mensual', 'AVG', 'UP', 'Porcentaje de residuos reciclados'),
    (6, 'Satisfacción Laboral', 'S', 'Clima', 'puntos', 'Promedio encuestas', 'Importante', 'Trimestral', 'AVG', 'UP', 'Satisfacción general (1-10)'),
    (7, 'Horas Capacitación', 'S', 'Desarrollo', 'horas', 'Suma horas capacitación', 'Secundaria', 'Mensual', 'SUM', 'UP', 'Horas de formación'),
    (8, 'Días Baja Accidentes', 'S', 'Seguridad', 'días', 'Suma días baja por accidente', 'Crítica', 'Mensual', 'SUM', 'DOWN', 'Días perdidos por accidentes'),
    (9, 'Margen Neto', 'Financiera', 'Rentabilidad', '%', '(Ingresos - Costos - Gastos)/Ingresos * 100', 'Crítica', 'Mensual', 'AVG', 'UP', 'Rentabilidad final'),
    (10, 'Huella Carbono', 'E', 'Emisiones', 'tCO2e', 'Consumo_energético * factor_emision', 'Crítica', 'Mensual', 'SUM', 'DOWN', 'Emisiones estimadas (factor 0.4 kgCO2/kWh)'),
    (11, 'Intensidad Energética', 'E', 'Eficiencia', 'kWh/ARS', 'Consumo_energético / Ingresos', 'Importante', 'Mensual', 'AVG', 'DOWN', 'Eficiencia energética por peso'),
    (12, 'Productividad Empleado', 'Financiera', 'Productividad', 'ARS/empl', 'Ingresos / N° empleados', 'Importante', 'Mensual', 'AVG', 'UP', 'Ingreso por empleado'),
]

columnas_metrica = ['id_metrica', 'nombre', 'categoria', 'subcategoria', 'unidad', 'formula', 
                    'prioridad', 'frecuencia_recomendada', 'tipo_agregacion', 'tendencia_deseada', 'descripcion']

dim_metrica = pd.DataFrame(metricas, columns=columnas_metrica)
logger.info(f"✅ dim_metrica: {len(dim_metrica)} métricas definidas")

2026-04-15 19:35:07,119 - INFO - 
2026-04-15 19:35:07,128 - INFO - CREANDO DIMENSIÓN MÉTRICA (ampliada)
2026-04-15 19:35:07,141 - INFO - ==================================================
2026-04-15 19:35:07,164 - INFO - ✅ dim_metrica: 12 métricas definidas


In [121]:
# logger.info("\n" + "="*50)
# logger.info("CREANDO DIMENSIÓN MÉTRICA")
# logger.info("="*50)

# metricas = [
#     (1, 'Ingresos', 'Financiera', 'ARS', 'SUM', 'UP', 'Suma de ventas'),
#     (2, 'Costo Compras', 'Financiera', 'ARS', 'SUM', 'DOWN', 'Suma de compras'),
#     (3, 'Gasto Personal', 'Financiera', 'ARS', 'SUM', 'DOWN', 'Suma de nóminas'),
#     (4, 'Consumo Energético', 'E', 'kWh', 'SUM', 'DOWN', 'Electricidad + gas'),
#     (5, 'Tasa Reciclaje', 'E', '%', 'AVG', 'UP', 'Reciclado / Generado'),
#     (6, 'Satisfacción Laboral', 'S', 'puntos', 'AVG', 'UP', 'Promedio encuestas'),
#     (7, 'Horas Capacitación', 'S', 'horas', 'SUM', 'UP', 'Horas de formación'),
#     (8, 'Días Baja Accidentes', 'S', 'días', 'SUM', 'DOWN', 'Días perdidos por accidentes'),
#     (9, 'Margen Neto', 'Financiera', '%', 'AVG', 'UP', '((Ingresos - Costos)/Ingresos)*100'),
#     (10, 'Huella Carbono', 'E', 'tCO2e', 'SUM', 'DOWN', 'Consumo * factor emisión'),
#     (11, 'Intensidad Energética', 'E', 'kWh/ARS', 'AVG', 'DOWN', 'Consumo / Ingresos'),
#     (12, 'Productividad Empleado', 'Financiera', 'ARS/empl', 'AVG', 'UP', 'Ingresos / N° empleados'),
# ]
# dim_metrica = pd.DataFrame(metricas, columns=['id_metrica', 'nombre', 'categoria', 'unidad', 'tipo_agregacion', 'tendencia_deseada', 'descripcion'])
# logger.info(f"✅ dim_metrica: {len(dim_metrica)} KPIs definidos")

#### <font color='lightgreen'>5.7 Dimensión Objetivo (metas anuales)</font>

In [122]:
logger.info("\n" + "="*50)
logger.info("CREANDO DIMENSIÓN OBJETIVO")
logger.info("="*50)

objetivos = [
    (1, 2024, 100000000, 95000000, 85000000, 0.10),
    (2, 2024, 60000000, 65000000, 70000000, 0.05),
    # ... completar con los 12 registros como en el original ...
]
dim_objetivo = pd.DataFrame(objetivos, columns=['id_metrica', 'anio', 'valor_objetivo', 'umbral_verde', 'umbral_amarillo', 'peso_relativo'])
dim_objetivo.insert(0, 'id_objetivo', range(1, len(dim_objetivo)+1))
logger.info(f"✅ dim_objetivo: {len(dim_objetivo)} metas anuales")

2026-04-15 19:35:07,695 - INFO - 
2026-04-15 19:35:07,704 - INFO - CREANDO DIMENSIÓN OBJETIVO
2026-04-15 19:35:07,713 - INFO - ==================================================
2026-04-15 19:35:07,734 - INFO - ✅ dim_objetivo: 2 metas anuales


### <font color='lightgreen'>6. Construir tabla de hechos fact_monitoreo</font>

In [123]:
logger.info("\n" + "="*50)
logger.info("CONSTRUYENDO FACT_MONITOREO")
logger.info("="*50)

mediciones = []
id_monitoreo = 1
mapa_id_tiempo = dict(zip(dim_tiempo['fecha'], dim_tiempo['id_tiempo']))

# Función auxiliar para agregar medición
def add_medicion(id_tiempo, id_metrica, valor, id_area=1):
    global id_monitoreo
    if pd.notna(valor) and id_tiempo in mapa_id_tiempo:
        mediciones.append([id_monitoreo, id_tiempo, id_metrica, id_area, valor])
        id_monitoreo += 1

2026-04-15 19:35:08,159 - INFO - 
2026-04-15 19:35:08,167 - INFO - CONSTRUYENDO FACT_MONITOREO
2026-04-15 19:35:08,177 - INFO - ==================================================


#### <font color='lightgreen'>6.1 Ingresos (id_metrica=1)</font>

In [124]:
df = datos['ventas']
ingresos = df.groupby(pd.Grouper(key='fecha', freq='ME'))['subtotal_ars'].sum().reset_index()
for _, row in ingresos.iterrows():
    id_t = mapa_id_tiempo.get(row['fecha'].replace(day=1))
    add_medicion(id_t, 1, row['subtotal_ars'])
logger.info(f"   ✅ Ingresos: {len(ingresos)} registros mensuales")

2026-04-15 19:35:09,475 - INFO -    ✅ Ingresos: 96 registros mensuales


#### <font color='lightgreen'>6.2 Costo Compras (id_metrica=2)</font>

In [125]:
df = datos['compras']
compras = df.groupby(pd.Grouper(key='fecha_emision', freq='ME'))['monto_ars'].sum().reset_index()
for _, row in compras.iterrows():
    id_t = mapa_id_tiempo.get(row['fecha_emision'].replace(day=1))
    add_medicion(id_t, 2, row['monto_ars'])
logger.info(f"   ✅ Costo Compras: {len(compras)} registros mensuales")

2026-04-15 19:35:09,769 - INFO -    ✅ Costo Compras: 96 registros mensuales


#### <font color='lightgreen'>6.3 Gasto Personal (id_metrica=3)</font>

In [126]:
df = datos['personal_nomina']
nomina = df.groupby(pd.Grouper(key='mes', freq='ME'))['total_devengado'].sum().reset_index()
for _, row in nomina.iterrows():
    id_t = mapa_id_tiempo.get(row['mes'].replace(day=1))
    add_medicion(id_t, 3, row['total_devengado'])
logger.info(f"   ✅ Gasto Personal: {len(nomina)} registros mensuales")

2026-04-15 19:35:10,104 - INFO -    ✅ Gasto Personal: 96 registros mensuales


#### <font color='lightgreen'>6.4 Consumo Energético (id_metrica=4)</font>

In [127]:
df = datos['consumo_energetico']
col_energia = 'kwh_totales' if 'kwh_totales' in df.columns else 'kwh_consumidos'
energia = df.groupby(pd.Grouper(key='fecha', freq='ME'))[col_energia].sum().reset_index()
for _, row in energia.iterrows():
    id_t = mapa_id_tiempo.get(row['fecha'].replace(day=1))
    add_medicion(id_t, 4, row[col_energia])
logger.info(f"   ✅ Consumo Energético: {len(energia)} registros mensuales")

2026-04-15 19:35:10,807 - INFO -    ✅ Consumo Energético: 96 registros mensuales


#### <font color='lightgreen'>6.5 Tasa Reciclaje (id_metrica=5)</font>

In [128]:
df = datos['residuos']
reciclaje = df.groupby(pd.Grouper(key='fecha_semana', freq='ME'))['tasa_reciclaje'].mean().reset_index()
for _, row in reciclaje.iterrows():
    id_t = mapa_id_tiempo.get(row['fecha_semana'].replace(day=1))
    add_medicion(id_t, 5, row['tasa_reciclaje'])
logger.info(f"   ✅ Tasa Reciclaje: {len(reciclaje)} registros mensuales")

2026-04-15 19:35:11,499 - INFO -    ✅ Tasa Reciclaje: 96 registros mensuales


#### <font color='lightgreen'>6.6 Satisfacción Laboral (id_metrica=6)</font>

In [129]:
df = datos['encuestas']
sat = df.groupby(pd.Grouper(key='fecha_encuesta', freq='QE'))['satisfaccion_gral'].mean().reset_index()
for _, row in sat.iterrows():
    id_t = mapa_id_tiempo.get(row['fecha_encuesta'].replace(day=1))
    add_medicion(id_t, 6, row['satisfaccion_gral'])
logger.info(f"   ✅ Satisfacción Laboral: {len(sat)} registros trimestrales")

2026-04-15 19:35:12,083 - INFO -    ✅ Satisfacción Laboral: 31 registros trimestrales


#### <font color='lightgreen'>6.7 Horas Capacitación (id_metrica=7)</font>

In [130]:
df = datos['eventos_rrhh']
cap = df[df['tipo_evento'] == 'Capacitación']
horas = cap.groupby(pd.Grouper(key='fecha', freq='ME'))['horas_capacitacion'].sum().reset_index()
for _, row in horas.iterrows():
    id_t = mapa_id_tiempo.get(row['fecha'].replace(day=1))
    add_medicion(id_t, 7, row['horas_capacitacion'])
logger.info(f"   ✅ Horas Capacitación: {len(horas)} registros mensuales")

2026-04-15 19:35:12,119 - INFO -    ✅ Horas Capacitación: 96 registros mensuales


#### <font color='lightgreen'>6.8 Días Baja Accidentes (id_metrica=8)</font>

In [131]:
acc = df[df['tipo_evento'] == 'Accidente Laboral']
dias = acc.groupby(pd.Grouper(key='fecha', freq='ME'))['dias_baja'].sum().reset_index()
for _, row in dias.iterrows():
    id_t = mapa_id_tiempo.get(row['fecha'].replace(day=1))
    add_medicion(id_t, 8, row['dias_baja'])
logger.info(f"   ✅ Días Baja Accidentes: {len(dias)} registros mensuales")

2026-04-15 19:35:12,156 - INFO -    ✅ Días Baja Accidentes: 96 registros mensuales


#### <font color='lightgreen'>6.9 Métricas derivadas (Margen Neto, Huella Carbono, Intensidad, Productividad)</font>

In [132]:
# Primero, obtener número de empleados por mes
empleados_mes = datos['personal_nomina'].groupby(pd.Grouper(key='mes', freq='ME'))['id_empleado'].nunique().reset_index()
empleados_mes['id_tiempo'] = empleados_mes['mes'].dt.strftime('%Y%m%d').astype(int)
empleados_dict = dict(zip(empleados_mes['id_tiempo'], empleados_mes['id_empleado']))

# Convertir mediciones actuales a DataFrame para pivotear
cols = ['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor']
df_med = pd.DataFrame(mediciones, columns=cols)
pivot = df_med.pivot_table(index='id_tiempo', columns='id_metrica', values='valor', aggfunc='first').reset_index()

for _, row in pivot.iterrows():
    id_t = row['id_tiempo']
    ingresos = row.get(1, 0.0)
    costo = row.get(2, 0.0)
    gasto = row.get(3, 0.0)
    consumo = row.get(4, 0.0)
    num_emp = empleados_dict.get(id_t, 1)

    # Margen Neto (9)
    margen = ((ingresos - costo - gasto) / ingresos * 100) if ingresos > 0 else 0.0
    add_medicion(id_t, 9, margen)

    # Huella Carbono (10)
    huella = consumo * FACTOR_EMISION
    add_medicion(id_t, 10, huella)

    # Intensidad Energética (11)
    intensidad = consumo / ingresos if ingresos > 0 else 0.0
    add_medicion(id_t, 11, intensidad)

    # Productividad (12)
    productividad = ingresos / num_emp if num_emp > 0 else 0.0
    add_medicion(id_t, 12, productividad)

logger.info(f"   ✅ Métricas derivadas: Margen Neto, Huella Carbono, Intensidad Energética, Productividad")

2026-04-15 19:35:12,206 - INFO -    ✅ Métricas derivadas: Margen Neto, Huella Carbono, Intensidad Energética, Productividad


In [133]:
# Verificar que los datasets tienen datos
print("Ventas:", len(datos['ventas']))
print("Compras:", len(datos['compras']))
print("Personal_nomina:", len(datos['personal_nomina']))
print("Consumo_energetico:", len(datos['consumo_energetico']))

# Verificar que las fechas sean datetime y tengan valores
print("\nTipo de columna fecha en ventas:", datos['ventas']['fecha'].dtype)
print("Primeras fechas en ventas:", datos['ventas']['fecha'].head(3).tolist())

# Verificar el mapa de tiempo
print("\nPrimeros 3 id_tiempo en dim_tiempo:", dim_tiempo[['id_tiempo', 'fecha']].head(3))

# Probar una agregación simple de ingresos
ingresos = datos['ventas'].groupby(pd.Grouper(key='fecha', freq='ME'))['subtotal_ars'].sum().reset_index()
print("\nFilas de ingresos agrupadas:", len(ingresos))
if len(ingresos) > 0:
    print("Primera fila:", ingresos.iloc[0])
    fecha_key = ingresos.iloc[0]['fecha'].replace(day=1)
    id_t = dict(zip(dim_tiempo['fecha'], dim_tiempo['id_tiempo'])).get(fecha_key)
    print("¿Fecha encontrada en dim_tiempo?", id_t is not None)
else:
    print("No hay ingresos agrupables (ventas vacías o fechas nulas)")

Ventas: 43893
Compras: 864
Personal_nomina: 1378
Consumo_energetico: 178

Tipo de columna fecha en ventas: datetime64[us]
Primeras fechas en ventas: [Timestamp('2018-01-01 00:00:00'), Timestamp('2018-01-01 00:00:00'), Timestamp('2018-01-01 00:00:00')]

Primeros 3 id_tiempo en dim_tiempo:    id_tiempo      fecha
0   20180101 2018-01-01
1   20180102 2018-01-02
2   20180103 2018-01-03

Filas de ingresos agrupadas: 96
Primera fila: fecha           2018-01-31 00:00:00
subtotal_ars            21202242.14
Name: 0, dtype: object
¿Fecha encontrada en dim_tiempo? True


### <font color='lightgreen'>7. Construir DataFrame final fact_monitoreo</font>

In [134]:
# ============================================
# CONSTRUCCIÓN DE fact_monitoreo (versión directa)
# ============================================

mediciones = []
id_monitoreo = 1

# Mapa de fecha a id_tiempo
mapa_id_tiempo = {fecha: id_t for fecha, id_t in zip(dim_tiempo['fecha'], dim_tiempo['id_tiempo'])}

# 1. Ingresos (id_metrica=1)
print("Procesando Ingresos...")
ingresos = datos['ventas'].groupby(pd.Grouper(key='fecha', freq='ME'))['subtotal_ars'].sum().reset_index()
for _, row in ingresos.iterrows():
    fecha_mes = row['fecha'].replace(day=1)
    id_t = mapa_id_tiempo.get(fecha_mes)
    if id_t:
        mediciones.append([id_monitoreo, id_t, 1, 1, row['subtotal_ars'], 'ventas'])
        id_monitoreo += 1
print(f"  → {len(ingresos)} registros, mediciones ahora: {len(mediciones)}")

# 2. Costo Compras (id_metrica=2)
print("Procesando Costo Compras...")
compras = datos['compras'].groupby(pd.Grouper(key='fecha_emision', freq='ME'))['monto_ars'].sum().reset_index()
for _, row in compras.iterrows():
    fecha_mes = row['fecha_emision'].replace(day=1)
    id_t = mapa_id_tiempo.get(fecha_mes)
    if id_t:
        mediciones.append([id_monitoreo, id_t, 2, 1, row['monto_ars'], 'compras'])
        id_monitoreo += 1
print(f"  → {len(compras)} registros, mediciones ahora: {len(mediciones)}")

# 3. Gasto Personal (id_metrica=3)
print("Procesando Gasto Personal...")
nomina = datos['personal_nomina'].groupby(pd.Grouper(key='mes', freq='ME'))['total_devengado'].sum().reset_index()
for _, row in nomina.iterrows():
    fecha_mes = row['mes'].replace(day=1)
    id_t = mapa_id_tiempo.get(fecha_mes)
    if id_t:
        mediciones.append([id_monitoreo, id_t, 3, 1, row['total_devengado'], 'nomina'])
        id_monitoreo += 1
print(f"  → {len(nomina)} registros, mediciones ahora: {len(mediciones)}")

# 4. Consumo Energético (id_metrica=4)
print("Procesando Consumo Energético...")
df_energia = datos['consumo_energetico']
col_energia = 'kwh_totales' if 'kwh_totales' in df_energia.columns else 'kwh_consumidos'
energia = df_energia.groupby(pd.Grouper(key='fecha', freq='ME'))[col_energia].sum().reset_index()
for _, row in energia.iterrows():
    fecha_mes = row['fecha'].replace(day=1)
    id_t = mapa_id_tiempo.get(fecha_mes)
    if id_t:
        mediciones.append([id_monitoreo, id_t, 4, 1, row[col_energia], 'energia'])
        id_monitoreo += 1
print(f"  → {len(energia)} registros, mediciones ahora: {len(mediciones)}")

# 5. Tasa Reciclaje (id_metrica=5)
print("Procesando Tasa Reciclaje...")
reciclaje = datos['residuos'].groupby(pd.Grouper(key='fecha_semana', freq='ME'))['tasa_reciclaje'].mean().reset_index()
for _, row in reciclaje.iterrows():
    fecha_mes = row['fecha_semana'].replace(day=1)
    id_t = mapa_id_tiempo.get(fecha_mes)
    if id_t:
        mediciones.append([id_monitoreo, id_t, 5, 1, row['tasa_reciclaje'], 'residuos'])
        id_monitoreo += 1
print(f"  → {len(reciclaje)} registros, mediciones ahora: {len(mediciones)}")

# 6. Satisfacción Laboral (id_metrica=6)
print("Procesando Satisfacción Laboral...")
sat = datos['encuestas'].groupby(pd.Grouper(key='fecha_encuesta', freq='QE'))['satisfaccion_gral'].mean().reset_index()
for _, row in sat.iterrows():
    fecha_mes = row['fecha_encuesta'].replace(day=1)
    id_t = mapa_id_tiempo.get(fecha_mes)
    if id_t:
        mediciones.append([id_monitoreo, id_t, 6, 1, row['satisfaccion_gral'], 'encuestas'])
        id_monitoreo += 1
print(f"  → {len(sat)} registros, mediciones ahora: {len(mediciones)}")

# 7. Horas Capacitación (id_metrica=7)
print("Procesando Horas Capacitación...")
df_eventos = datos['eventos_rrhh']
cap = df_eventos[df_eventos['tipo_evento'] == 'Capacitación']
if not cap.empty:
    horas = cap.groupby(pd.Grouper(key='fecha', freq='ME'))['horas_capacitacion'].sum().reset_index()
    for _, row in horas.iterrows():
        fecha_mes = row['fecha'].replace(day=1)
        id_t = mapa_id_tiempo.get(fecha_mes)
        if id_t:
            mediciones.append([id_monitoreo, id_t, 7, 1, row['horas_capacitacion'], 'eventos_rrhh'])
            id_monitoreo += 1
    print(f"  → {len(horas)} registros, mediciones ahora: {len(mediciones)}")
else:
    print("  → No hay eventos de capacitación")

# 8. Días Baja Accidentes (id_metrica=8)
print("Procesando Días Baja Accidentes...")
acc = df_eventos[df_eventos['tipo_evento'] == 'Accidente Laboral']
if not acc.empty:
    dias = acc.groupby(pd.Grouper(key='fecha', freq='ME'))['dias_baja'].sum().reset_index()
    for _, row in dias.iterrows():
        fecha_mes = row['fecha'].replace(day=1)
        id_t = mapa_id_tiempo.get(fecha_mes)
        if id_t:
            mediciones.append([id_monitoreo, id_t, 8, 1, row['dias_baja'], 'eventos_rrhh'])
            id_monitoreo += 1
    print(f"  → {len(dias)} registros, mediciones ahora: {len(mediciones)}")
else:
    print("  → No hay eventos de accidentes")

# 9. Número de empleados por mes (para derivadas)
empleados_mes = datos['personal_nomina'].groupby(pd.Grouper(key='mes', freq='ME'))['id_empleado'].nunique().reset_index()
empleados_mes['id_tiempo'] = empleados_mes['mes'].dt.strftime('%Y%m%d').astype(int)
empleados_dict = dict(zip(empleados_mes['id_tiempo'], empleados_mes['id_empleado']))

# 10. Métricas derivadas
print("Procesando métricas derivadas...")
if mediciones:
    # Crear DataFrame temporal para pivotear
    cols = ['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor', 'fuente']
    df_temp = pd.DataFrame(mediciones, columns=cols)
    pivot = df_temp.pivot_table(index='id_tiempo', columns='id_metrica', values='valor', aggfunc='first').reset_index()
    
    FACTOR_EMISION = 0.0004  # tCO2e/kWh
    for _, row in pivot.iterrows():
        id_t = row['id_tiempo']
        ingresos = row.get(1, 0.0)
        costo = row.get(2, 0.0)
        gasto = row.get(3, 0.0)
        consumo = row.get(4, 0.0)
        num_emp = empleados_dict.get(id_t, 1)
        
        # Margen Neto (9)
        margen = ((ingresos - costo - gasto) / ingresos * 100) if ingresos > 0 else 0.0
        mediciones.append([id_monitoreo, id_t, 9, 1, margen, 'derivada'])
        id_monitoreo += 1
        
        # Huella Carbono (10)
        huella = consumo * FACTOR_EMISION
        mediciones.append([id_monitoreo, id_t, 10, 1, huella, 'derivada'])
        id_monitoreo += 1
        
        # Intensidad Energética (11)
        intensidad = consumo / ingresos if ingresos > 0 else 0.0
        mediciones.append([id_monitoreo, id_t, 11, 1, intensidad, 'derivada'])
        id_monitoreo += 1
        
        # Productividad (12)
        productividad = ingresos / num_emp if num_emp > 0 else 0.0
        mediciones.append([id_monitoreo, id_t, 12, 1, productividad, 'derivada'])
        id_monitoreo += 1
    print(f"  → Derivadas agregadas, total mediciones: {len(mediciones)}")
else:
    print("  → No hay mediciones base para derivadas")

# Construir DataFrame final
if mediciones:
    fact_monitoreo = pd.DataFrame(mediciones, columns=['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor', 'fuente'])
    fact_monitoreo = fact_monitoreo.sort_values(['id_tiempo', 'id_metrica']).reset_index(drop=True)
    print(f"\n✅ fact_monitoreo generado con {len(fact_monitoreo)} registros")
else:
    print("\n❌ No se generaron mediciones. Revisar datos de origen.")
    fact_monitoreo = pd.DataFrame(columns=['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor', 'fuente'])

# Guardar a CSV
fact_monitoreo.to_csv(DATA_CURATED / "fact_monitoreo.csv", index=False)
print(f"💾 Guardado en {DATA_CURATED / 'fact_monitoreo.csv'}")

Procesando Ingresos...
  → 96 registros, mediciones ahora: 96
Procesando Costo Compras...
  → 96 registros, mediciones ahora: 192
Procesando Gasto Personal...
  → 96 registros, mediciones ahora: 288
Procesando Consumo Energético...
  → 96 registros, mediciones ahora: 384
Procesando Tasa Reciclaje...
  → 96 registros, mediciones ahora: 480
Procesando Satisfacción Laboral...
  → 31 registros, mediciones ahora: 511
Procesando Horas Capacitación...
  → 96 registros, mediciones ahora: 607
Procesando Días Baja Accidentes...
  → 96 registros, mediciones ahora: 703
Procesando métricas derivadas...
  → Derivadas agregadas, total mediciones: 1087

✅ fact_monitoreo generado con 1087 registros
💾 Guardado en c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\curated\fact_monitoreo.csv


In [135]:
# fact_monitoreo = pd.DataFrame(mediciones, columns=['id_monitoreo', 'id_tiempo', 'id_metrica', 'id_area', 'valor'])
# fact_monitoreo = fact_monitoreo.sort_values(['id_tiempo', 'id_metrica']).reset_index(drop=True)

# # Validar integridad referencial: todos los id_tiempo deben existir en dim_tiempo
# ids_validos = set(dim_tiempo['id_tiempo'])
# ids_en_hechos = set(fact_monitoreo['id_tiempo'])
# missing = ids_en_hechos - ids_validos
# if missing:
#     logger.warning(f"⚠️ {len(missing)} id_tiempo sin correspondencia en dim_tiempo. Eliminando...")
#     fact_monitoreo = fact_monitoreo[fact_monitoreo['id_tiempo'].isin(ids_validos)]

# logger.info(f"\n✅ fact_monitoreo: {len(fact_monitoreo)} registros")
# logger.info(f"   Métricas incluidas: {fact_monitoreo['id_metrica'].nunique()}")
# logger.info(f"   Períodos: {fact_monitoreo['id_tiempo'].nunique()}")

### <font color='lightgreen'>8. Guardar modelo estrella</font>

In [136]:
logger.info("\n" + "="*50)
logger.info("GUARDANDO MODELO ESTRELLA")
logger.info("="*50)

# Dimensiones
dimensiones = {
    'dim_tiempo': dim_tiempo,
    'dim_area': dim_area,
    'dim_empleado': dim_empleado,
    'dim_proveedor': dim_proveedor,
    'dim_categoria': dim_categoria,
    'dim_metrica': dim_metrica,
    'dim_objetivo': dim_objetivo
}

for nombre, df in dimensiones.items():
    out_path = DATA_CURATED / f"{nombre}.csv"
    df.to_csv(out_path, index=False, encoding='utf-8')
    logger.info(f"   💾 {nombre}.csv ({len(df)} filas)")

# Hechos
fact_monitoreo.to_csv(DATA_CURATED / "fact_monitoreo.csv", index=False, encoding='utf-8')
logger.info(f"   💾 fact_monitoreo.csv ({len(fact_monitoreo)} filas)")

2026-04-15 19:35:12,504 - INFO - 
2026-04-15 19:35:12,504 - INFO - GUARDANDO MODELO ESTRELLA
2026-04-15 19:35:12,505 - INFO - ==================================================
2026-04-15 19:35:12,520 - INFO -    💾 dim_tiempo.csv (2641 filas)
2026-04-15 19:35:12,524 - INFO -    💾 dim_area.csv (5 filas)
2026-04-15 19:35:12,530 - INFO -    💾 dim_empleado.csv (25 filas)
2026-04-15 19:35:12,534 - INFO -    💾 dim_proveedor.csv (5 filas)
2026-04-15 19:35:12,537 - INFO -    💾 dim_categoria.csv (10 filas)
2026-04-15 19:35:12,542 - INFO -    💾 dim_metrica.csv (12 filas)
2026-04-15 19:35:12,547 - INFO -    💾 dim_objetivo.csv (2 filas)
2026-04-15 19:35:12,553 - INFO -    💾 fact_monitoreo.csv (1087 filas)


### <font color='lightgreen'>9. Resumen final</font>

In [137]:
print("\n" + "="*50)
print("✅ MODELO ESTRELLA COMPLETADO")
print("="*50)
print("\n📊 Dimensiones creadas:")
for nombre, df in dimensiones.items():
    print(f"   • {nombre}: {len(df):,} registros")
print(f"\n📊 Tabla de Hechos:")
print(f"   • fact_monitoreo: {len(fact_monitoreo):,} registros, {len(fact_monitoreo.columns)} columnas")
print(f"\n📁 Archivos guardados en: {DATA_CURATED}/")
print("\n✅ Listo para cargar a DuckDB y conectar con el dashboard")


✅ MODELO ESTRELLA COMPLETADO

📊 Dimensiones creadas:
   • dim_tiempo: 2,641 registros
   • dim_area: 5 registros
   • dim_empleado: 25 registros
   • dim_proveedor: 5 registros
   • dim_categoria: 10 registros
   • dim_metrica: 12 registros
   • dim_objetivo: 2 registros

📊 Tabla de Hechos:
   • fact_monitoreo: 1,087 registros, 6 columnas

📁 Archivos guardados en: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\curated/

✅ Listo para cargar a DuckDB y conectar con el dashboard


In [138]:
# Deberían existir 8 archivos en data/curated/
assert len(list(DATA_CURATED.glob("*.csv"))) == 8

# fact_monitoreo no debe tener nulos en id_tiempo ni id_metrica
assert fact_monitoreo['id_tiempo'].isna().sum() == 0
assert fact_monitoreo['id_metrica'].isna().sum() == 0